# Лабораторная работа №1
## Разведочный анализ данных: Titanic

**Выполнил:** Акимов Александр, гр. 4374 

---

**Цель работы:** провести разведочный анализ данных набора данных 
Titanic, выявить закономерности, определить ключевые признаки и подготовить 
данные для построения модели машинного обучения.

**Задача:** бинарная классификация — предсказать, выжил ли пассажир 

**Источник данных:** [Kaggle — Titanic: Machine Learning from Disaster](https://www.kaggle.com/c/titanic)

## 1. Загрузка данных и первичный обзор

Загружаем обучающую выборку и смотрим на её структуру.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# читаем датасет из файла и выводим его шапку
df = pd.read_csv('data/train.csv')
display(df.head())

### 1.1 Первичный обзор
Проверяем размеры датасета, типы данных признаков, целевую переменную

In [ ]:
display(df.info())

### Вывод:

**Структура данных:**
- **891 строка** — пассажиры обучающей выборки;
- **12 столбцов** — характеристики;
- **Целевая переменная:** `Survived` (0 / 1).

**Столбцы:**

| Столбец | Тип | Описание |
|---|---|---|
| `PassengerId` | int | ID пассажира |
| `Survived` | int | **Целевая**: 0 — погиб, 1 — выжил |
| `Pclass` | int | Класс билета |
| `Name` | str | Имя пассажира |
| `Sex` | str | Пол пассажира |
| `Age` | float | Возраст |
| `SibSp` | int | Количество братьев/сестёр + супругов |
| `Parch` | int | Количество родителей/детей |
| `Ticket` | str | Номер билета |
| `Fare` | float | Стоимость билета |
| `Cabin` | str | Номер каюты |
| `Embarked` | str | Порт посадки |

## 2. Разведочный анализ
**Подготавливаем данные для обучения:**
- проверка качества данных
- целевая переменная
- одномерный анализ
- двумерный анализ
- корреляции
- комбинированные признаки
- выводы

### 2.1 Проверка качества данных
Проверяем наличие:
- пропусков
- дубликатов
- аномалий

In [ ]:
# Итоговая таблица пропусков
missing = {
    'Столбец': df.columns,
    'Пропусков': df.isnull().sum(),
    'Доля, %': (df.isnull().sum() / len(df)).round(3)
}

missing_df = pd.DataFrame(missing)
display(missing_df)

In [ ]:
# оставим только столбцы с пропусками 
missing_df = missing_df[missing_df['Пропусков'] > 0]

# отсортируем по убыванию
missing_df = missing_df.sort_values('Пропусков', ascending=False)
display(missing_df)

#### Вывод по пропускам:
- Для `Cabin` > 75% пропусков - удаляем признак (мало информации)
- Для `Age` менее 25% пропусков - заполняем медианой
- Для `Embarked` менее 1% - заполняем модой (категориальный признак)

In [ ]:
# провека дубликатов
print(f'Дубликатов: {df.duplicated().sum()}')

#### Вывод по дубликатам:
- дубликатов не обнаружено

#### Поиск аномальных значений:
Проверим отдельно **числовые** и **строковые** параметры.

Будем так же смотреть на кол-во **уникальных** значений параметра в поисках ошибок

In [ ]:
# разделим признаки на числовые и строковые
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
str_cols = df.select_dtypes(include=['str']).columns

# проверка аномалий числовых значений
display(df[num_cols].describe().round(3))

In [ ]:
# проверим уникальные значения
display(df[num_cols].nunique())

# выведем интересующие уникальные значения
for col in ['Survived', 'Pclass']:
    uniques = sorted(df[col].dropna().unique().tolist())
    print(f"{col}: уникальных = {uniques}")

#### Вывод по числовым параметрам:
- кол-во ID равно кол-ву наблюдений, в диапазоне от 1 до 891 - **без аномалий**
- кол-во уникальных значений `Survived` - 2: выжил или нет - **без аномалий**
- кол-во униклаьных значений `Pclass` - 3: 1й, 2й, 3й - **без аномалий**
- диапазон возраста от 0.42 до 80 лет - **без аномалий**
- кол-во родственников в пределах разумного значения - от 0 до 8 - **без аномалий**
- стоимость билетов разнится, но без аномальных значений - **без аномалий**

In [ ]:
# проверка аномалий строковых значений
display(df[str_cols].nunique())

# выведем интересующие уникальные значения
for col in ['Sex', 'Embarked']:
    uniques = sorted(df[col].dropna().unique().tolist())
    print(f"{col}: уникальных = {uniques}")

#### Вывод по строковым параметрам:
- кол-во имен равно кол-ву наблюдений - **без аномалий**
- кол-во уникальных значений `Sex` - 2: мужчина или женщина - **без аномалий**
- кол-во униклаьных значений `Embarked` - 3: C, Q, S - **без аномалий**

### 2.2. Целевая переменная

Целевая переменная — `Survived` (0 — погиб, 1 — выжил).

Проверим:
- распределение классов
- наличие дисбаланса

In [ ]:
# распределение целевой переменной
counts = df['Survived'].value_counts()

# таблица распределения
survived_dist = {
    'Класс': ['0 (Погиб)', '1 (Выжил)'],
    'Количество': [counts[0], counts[1]],
    'Доля, %': [round(counts[0] / len(df) * 100, 2), round(counts[1] / len(df) * 100, 2)]
}

survived_df = pd.DataFrame(survived_dist)
survived_df = survived_df.set_index('Класс')
display(survived_df)

# график распределения
sns.countplot(data=df, x='Survived')
plt.show()

#### Вывод по целевой переменной:
Дисбаланс = 1.6 - **умеренный**

Для разделения на train/test необходима **стратификация**, чтобы сохранить пропорции классов.

### 2.3 Одномерный анализ

Изучаем каждый признак отдельно:
- распределение
- диапазон
- выбросы

#### **Числовые признаки** - `Age`, `Fare`, `SibSp`, `Parch`.

Изучаем каждый признак отдельно: 
- распределение
- диапазон
- выбросы

Построим для каждого признака по 2 диаграммы:
- гистограмма распределения (histplot + kde)
- диаграмма выбросов (boxplot)

In [ ]:
# Числовые признаки для анализа
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']

# сетка графиков
fig, axes = plt.subplots(nrows=len(numeric_cols), ncols=2, figsize=(12, 4 * len(numeric_cols)))

# по 2 диаграммы для каждого признака
for i, col in enumerate(numeric_cols):

    # считаем статистики
    mean = df[col].mean()
    median = df[col].median()
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    
    # гистограмма распределения
    sns.histplot(data=df, x=col, kde=True, bins=30, ax=axes[i, 0])
    axes[i, 0].set_title(f'{col}: распределение')
    axes[i, 0].set_xlabel('')
    axes[i, 0].set_ylabel('Количество')
    axes[i, 0].axvline(mean, color='red', label=f'Среднее: {mean:.2f}')
    axes[i, 0].axvline(median, color='green', label=f'Медиана: {median:.2f}')
    axes[i, 0].legend()
    
    # boxplot — выбросы и квартили
    sns.boxplot(data=df, y=col, ax=axes[i, 1])
    axes[i, 1].set_title(f'{col}: boxplot')
    axes[i, 1].set_ylabel('')
    axes[i, 1].axhline(q3, color='orange', linestyle=':', label=f'75%: {q3:.2f}')
    axes[i, 1].axhline(q1, color='orange', linestyle=':', label=f'25%: {q1:.2f}')
    axes[i, 1].legend()
    
plt.suptitle('Анализ числовых признаков', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

#### Выводы по числовым признакам

**Возраст:**
- **Форма:** распределение близко к нормальному, но слегка скошено вправо - основная масса людей в возрасте 20–40 лет, но есть «хвост» из пожилых людей.
- **Boxplot:** есть выбросы сверху - люди старше 65 лет.

**Стоимость билета:**
- **Форма:** распределение сильно скошено вправо - очень много дешевых билетов и очень мало дорогих.
- **Boxplot:** много выбросов сверху (дорогие билеты 1-го класса).

**Количество братьев/сестёр и родителей/детей:**
- **Форма:** распределения дискретные и сильно скошенные вправо. Большинство пассажиров ехали без родственников.
- **Boxplot:** ящик и усы нулевые. Все ненулевые значения (хотя бы один родственник) являются выбросами.

Для линейных моделей к `Fare` стоит применить **логарифмирование**, чтобы уменьшить влияние выбросов и приблизить распределение к нормальному.

#### **Категориальные признаки** — `Sex`, `Pclass`, `Embarked`.

Изучаем каждый признак отдельно:
- распределение категорий (сколько человек в каждой группе)
- дисбаланс классов

Построим для каждого признака по 1 диаграмме:
- столбиковая диаграмма (`countplot` с подписями количества и долей).

In [ ]:
# категориальные признаки для анализа
categorical_cols = ['Sex', 'Pclass', 'Embarked']

for col in categorical_cols:
    
    # считаем частоты 
    counts = df[col].value_counts()
    
    # собираем данные
    data = {
        'Категория': counts.index.astype(str),
        'Количество': counts.values,
        'Доля, %': (counts.values / len(df) * 100).round(2)
    }
    
    # создаем DataFrame
    table = pd.DataFrame(data).set_index('Категория') 
    print(f"\nПризнак: {col}")
    display(table)

# сетка графиков
fig, axes = plt.subplots(nrows=len(categorical_cols), ncols=1, figsize=(10, 4 * len(categorical_cols)))

# по 1 диаграмме для каждого признака
for i, col in enumerate(categorical_cols):
    
    # строим countplot
    ax = sns.countplot(data=df, x=col, ax=axes[i])    
    axes[i].set_title(f'{col}: распределение', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Количество')

plt.suptitle('Анализ категориальных признаков', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

#### Выводы по категориальным признакам

**Пол:**
- В датасете мужчин почти в 2 раза больше, чем женщин - дисбаланс не критичный.

**Класс билета:**
- Распределение неравномерное: доминирует 3-й класс, меньше всего пассажиров 2-го класса.

**Порт посадки:**
- Большинство пассажиров сели в Саутгемптоне (`S`), значительно меньше в Шербуре (`C`) и Квинстауне (`Q`).

Признаки `Embarked` и `Sex` требуют **one-hot-encoding**.

### 2.4. Двумерный анализ

Изучаем связь каждого признака с целевой переменной `Survived`.

#### **Числовые признаки** - `Age`, `Fare`, `SibSp`, `Parch`.

Изучаем влияние признаков на выживаемость:
- Построим **boxplot** для сравнения медиан и разброса
- Построим **violinplot** для сравнения формы распределения

In [ ]:
# cредние значения числовых признаков по классам выживаемости
summary = df.groupby('Survived')[numeric_cols].mean().round(2)
print("Средние значения по классам выживаемости:")
display(summary)

# графики 
fig, axes = plt.subplots(nrows=len(numeric_cols), ncols=2, figsize=(14, 4 * len(numeric_cols)))

for i, col in enumerate(numeric_cols):
    
    # boxplot
    sns.boxplot(data=df, x='Survived', y=col, ax=axes[i, 0], hue='Survived')
    axes[i, 0].set_title(f'{col}: Boxplot по классам')
    axes[i, 0].set_xlabel('')
    axes[i, 0].set_ylabel('')
    
    # violinplot
    sns.violinplot(data=df, x='Survived', y=col, hue='Survived', ax=axes[i, 1])
    axes[i, 1].set_title(f'{col}: Violinplot по классам')
    axes[i, 1].set_xlabel('')
    axes[i, 1].set_ylabel('')

plt.suptitle('Связь числовых признаков с выживаемостью', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

#### Вывод по числовым признакам:

**Возраст:**
- Средний возраст выживших и погибших примерно одинаков.
- У выживших есть выраженный «второй пик» в области детей (0–10 лет).

**Стоимость билета:**
- Выжившие в среднем платили больше.

**Количество родственников:**
- У выживших чуть выше плотность в области значений 1–3. То есть люди с небольшой семьей выживали чаще, чем одиночки или люди с большими семьями.

`SibSp` и `Parch` лучше объединить в один признак `FamilySize`, т.к. они несут схожий смысл.

#### **Категориальные признаки** — `Sex`, `Pclass`, `Embarked`.

Изучаем влияние признаков на выживаемость:
- Построим **таблицы** и **диаграммы** с долями выживших

In [ ]:
# таблицы с долями выживших
for col in categorical_cols:
    
    # считаем долю выживших
    survival_rate = (df.groupby(col)['Survived'].mean() * 100).round(2)
    
    # оформляем в DataFrame
    summary = survival_rate.reset_index()
    summary.columns = [col, 'Доля выживших, %']
    
    print(f"\nВыживаемость по признаку '{col}':")
    display(summary.set_index(col))

# графики
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(16, 5))

for i, col in enumerate(categorical_cols):
    
    # countplot с разделением по Survived
    sns.countplot(data=df, x=col, hue='Survived', ax=axes[i])
    axes[i].set_title(f'Выживаемость по признаку: {col}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Количество')

plt.tight_layout()
plt.show()

#### Вывод по категориальным признакам:

**Пол:**
- Женщины выживали намного чаще, чем мужчины.

**Класс билета:**
- Чем выше класс, тем выше выживаемость.

**Порт посадки:**
- Пассажиры из Шербура выживали чаще, но сама по себе зависимость выживаемости от порта логики не несет. Скорее всего это связано с зависимостью порта посадки от других признаков

### 2.5. Корреляционный анализ

Строим матрицу корреляций для числовых признаков - проверяем признаки на **зависимость** и **мультиколлинеарность**

Для анализа исключим бесполезный `PassengerId`, а категориальные `Sex` и `Embarked` временно закодируем, чтобы увидеть их связь с целевой переменной.

In [ ]:
# создаем копию для анализа
df_corr = df.copy()

# удаляем ID
df_corr = df_corr.drop(columns=['PassengerId'])

# кодируем Embarked и Sex (One-Hot Encoding)
df_corr = pd.get_dummies(df_corr, columns=['Embarked', 'Sex'])

# cчитаем корреляцию только для числовых столбцов
corr_matrix = df_corr.select_dtypes(include=['int64', 'float64', 'bool']).corr()

# cтроим тепловую карту
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Матрица корреляций (с закодированными признаками)', fontsize=14)
plt.show()

# выводим корреляции с целевой переменной в виде таблицы
survived_corr = corr_matrix['Survived'].drop('Survived').sort_values(ascending=False)
corr_df = pd.DataFrame({
    'Признак': survived_corr.index,
    'Корреляция с Survived': survived_corr.values.round(3)
}).set_index('Признак')

display(corr_df)

#### Вывод по корреляциям:

**Связь с целевой переменной:**
- `Sex_female` — **сильнейшая** положительная корреляция. Женщины имели значительно больше шансов на спасение. Мужчины - наоборот.
- `Pclass` — **сильная** отрицательная корреляция. Чем больше номер класса, тем ниже шанс выжить.
- `Fare` — **умеренная** положительная корреляция. Чем дороже билет, тем выше шанс выжить.
- `Embarked_C` (порт Шербур) — **слабая** положительная связь. Пассажиры из этого порта чаще выживали.
- Остальные признаки имеют **очень слабую связь** с выживаемостью.

**Мультиколлинеарность (связи между признаками):**
- `Pclass` и `Fare` — **сильная** отрицательная связь. Билеты 1-го класса стоят дорого, 3-го — дешево.
- `SibSp` и `Parch` — **умеренная** положительная связь. если человек едет с семьей, оба парамета больше 0.
- `Embarked_C` и `Pclass` — **умеренная** отрицательная связь. Пассажиры из Шербура чаще покупали билеты 1-го класса.

**Итог:** 
- Признак `Sex` имеет самое сильное влияние на выживаемость.
- Из-за мультиколлинеарности `Pclass` и `Fare` для линейных моделей потребуется **отбор признаков**.
- Из-за парной связи `SibSp` и `Parch`, для линейных моделей потребуется **объединение** признаков.

### 2.6. Комбинированные признаки

Проверяем, дают ли комбинации признаков больше информации, чем отдельные признаки.

#### Логика комбинирования признаков

При создании новых признаков были выбраны следующие комбинации:

**`FamilySize` (Размер семьи) — объединение `SibSp` и `Parch`**
- **Причина:** В корреляционном анализе мы выявили умеренную положительную связь между `SibSp` и `Parch`. Оба признака описывают одно и то же — наличие семьи, но с разных сторон. Если оставить их оба, для линейных моделей возникнет проблема мультиколлинеарности.
- **Решение:** Сложить их в один признак `FamilySize = SibSp + Parch + 1`. Единица добавляется, чтобы учесть самого пассажира.

**`Sex_Pclass` (Пол + Класс) — взаимодействие двух сильных признаков**
- **Причина:** `Sex` и `Pclass` показали самую сильную связь с выживаемостью.
- **Решение:** Объединить их в строковый признак, затем закодировать.

In [ ]:
# создаем комбинированные признаки
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)

# анализ FamilySize
print("Выживаемость по размеру семьи:")
family_survival = (df.groupby('FamilySize')['Survived'].mean() * 100).round(2)
display(family_survival.to_frame('Доля выживших, %'))

# анализ Sex_Pclass
print("\nВыживаемость по комбинации Пол + Класс:")
sex_pclass_survival = (df.groupby('Sex_Pclass')['Survived'].mean() * 100).round(2).sort_values(ascending=False)
display(sex_pclass_survival.to_frame('Доля выживших, %'))

# графики
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

# barplot для FamilySize
sns.barplot(data=df, x='FamilySize', y='Survived', hue='FamilySize', ax=axes[0], errorbar=None, legend=False)
axes[0].set_title('Выживаемость по размеру семьи')
axes[0].set_ylabel('Доля выживших')

# barplot для Sex_Pclass
sns.barplot(data=df, x='Sex_Pclass', y='Survived', order=sex_pclass_survival.index, hue='Sex_Pclass', ax=axes[1], errorbar=None, legend=False)
axes[1].set_title('Выживаемость по Полу и Классу')
axes[1].set_ylabel('Доля выживших')

plt.tight_layout()
plt.show()

#### Вывод по комбинированным признакам:

**Размер семьи:**
- Пассажиры, путешествующие в одиночку, выживали в 30% случаев.
- Пассажиры с небольшой семьей выживали чаще всего.
- Пассажиры с большими семьями почти не спасались.

**Комбинация Пол + Класс:**
- Наивысший процент выживания у женщин 1-го и 2-го классов.
- Женщины 3-го класса выживали в 50% случаев.
- Мужчины 1-го класса - 37%, мужчины 3-го класса - всего 14%.
 
Комбинированные признаки несут больше информации, чем исходные. Признак `FamilySize` лучше заменит `SibSp` и `Parch`, а `Sex_Pclass` покажет скрытые закономерности. Эти признаки нужно будет создать на этапе **Feature Engineering**.

### 2.7. Выводы по EDA

На основе проведенного разведочного анализа данных можно сделать следующие выводы о датасете и рекомендации для дальнейшего моделирования.

#### 1. Качество данных
- **Пропуски:** `Cabin` (77%) — удалить; `Age` (20%) — заполнить медианой; `Embarked` (0.2%) — заполнить медианой.
- **Дубликаты:** отсутствуют.
- **Аномалии:** в числовых признаках не обнаружено критичных ошибок. Выбросы в `Fare` и `Age` являются реальными значениями.

#### 2. Целевая переменная
- `Survived` имеет умеренный дисбаланс (62% / 38%). 
- Для разделения на train/test обязательна **стратификация**.

#### 3. Ключевые признаки
1. `Sex` - сильнейший признак (женщины выживают чаще).
2. `Pclass` - сильный признак (1 класс выживает чаще).
3. `Fare` - связан с `Pclass` (дорогие билеты = 1 класс = выше шанс).
4. `Age` - дети выживали чаще, но в целом связь слабая.
5. `Embarked` - слабая связь (порт `C` выживал чаще, но это, скорее всего, связано с классом).

#### 4. Рекомендации по подготовке данных (Feature Engineering)
- **Удалить:** `PassengerId`, `Name`, `Ticket`, `Cabin`.
- **Заполнить пропуски:** `Age` (медиана), `Embarked` (мода).
- **Создать новые признаки:** 
    - `FamilySize` = `SibSp` + `Parch` + 1 (заменит два исходных).
    - `Sex_Pclass` - комбинация пола и класса
- **Закодировать:**
    - `Sex` (Label Encoding: male=0, female=1)
    - `Embarked` и `Sex_Pclass` (One-Hot Encoding).
- **Трансформировать:** для линейных моделей применить логарифмирование к `Fare`.
- **Отбор признаков:** для линейных моделей удалить `Fare` или `Pclass` (из-за мультиколлинеарности), а `SibSp` и `Parch` заменить на `FamilySize`.